# Decision Intelligence Evolution - Data Preparation

**Execution Order: 1 of 2**

This notebook handles:
1. Loading data from multiple sources (Scopus, ScienceDirect, Preprints)
2. Deduplication between Scopus and ScienceDirect
3. **Filtering to 2010+ (when Decision Intelligence was first developed)**
4. Filtering to Article, Review, and Conference Paper types
5. Merging all sources
6. Exporting integrated dataset **without abstracts** (for Git compatibility)

**Historical Context**: Dr. Lorien Pratt first coined and developed the term. By 2010, she and Mark Zangari founded the related concept of "decision engineering," which was officially formalized and renamed as Decision Intelligence (DI) around 2012.

**Next Step**: Run `02_visualization.ipynb` for analysis and visualizations

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

# Data directory
DATA_DIR = Path('data')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Libraries imported successfully")
print(f"✓ Data directory: {DATA_DIR.absolute()}")
print(f"✓ Output directory: {OUTPUT_DIR.absolute()}")

✓ Libraries imported successfully
✓ Data directory: /Users/wisu/Source/citrus-decision-intelligence/data
✓ Output directory: /Users/wisu/Source/citrus-decision-intelligence/output


## 2. Data Loading Functions

In [2]:
def parse_bibtex_file(filepath):
    """
    Parse a BibTeX file and extract key fields.
    Returns a list of dictionaries with publication data.
    """
    publications = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Split by @article or @inproceedings entries
    entries = re.split(r'@(?:article|inproceedings)\{', content)
    
    for entry in entries[1:]:  # Skip first empty split
        pub = {}
        
        # Extract citation key
        citation_match = re.match(r'([^,]+),', entry)
        if citation_match:
            pub['citation_key'] = citation_match.group(1)
        
        # Extract fields using regex
        fields = {
            'title': r'title\s*=\s*\{([^}]+)\}',
            'authors': r'author\s*=\s*\{([^}]+)\}',
            'year': r'year\s*=\s*\{([^}]+)\}',
            'journal': r'journal\s*=\s*\{([^}]+)\}',
            'doi': r'doi\s*=\s*\{([^}]+)\}',
            'abstract': r'abstract\s*=\s*\{([^}]+)\}',
            'keywords': r'keywords\s*=\s*\{([^}]+)\}',
        }
        
        for field, pattern in fields.items():
            match = re.search(pattern, entry, re.IGNORECASE | re.DOTALL)
            pub[field] = match.group(1).strip() if match else None
        
        publications.append(pub)
    
    return publications

print("✓ BibTeX parser function defined")

✓ BibTeX parser function defined


In [3]:
def load_scopus_data():
    """
    Load Scopus CSV data.
    """
    scopus_file = list(DATA_DIR.glob('*scopus*.csv'))[0]
    df = pd.read_csv(scopus_file, encoding='utf-8')
    
    # Standardize column names
    df = df.rename(columns={
        'Title': 'title',
        'Year': 'year',
        'Authors': 'authors',
        'Abstract': 'abstract',
        'Author Keywords': 'keywords',
        'Document Type': 'document_type',
        'DOI': 'doi',
        'Source title': 'journal'
    })
    
    # Add source flag
    df['source'] = 'Scopus'
    df['source_file'] = scopus_file.name
    
    return df

print("✓ Scopus loader function defined")

✓ Scopus loader function defined


In [4]:
def load_sciencedirect_data():
    """
    Load all ScienceDirect BibTeX files and extract document type from filename.
    """
    all_pubs = []
    
    # Find all BibTeX files
    bib_files = list(DATA_DIR.glob('*ScienceDirect*.bib'))
    
    for bib_file in bib_files:
        # Extract document type from filename
        filename = bib_file.stem
        
        if 'article' in filename.lower():
            doc_type = 'Article'
        elif 'review' in filename.lower():
            doc_type = 'Review'
        elif 'conference' in filename.lower():
            doc_type = 'Conference Paper'
        else:
            doc_type = 'Unknown'
        
        # Parse BibTeX file
        pubs = parse_bibtex_file(bib_file)
        
        # Add metadata
        for pub in pubs:
            pub['document_type'] = doc_type
            pub['source'] = 'ScienceDirect'
            pub['source_file'] = bib_file.name
        
        all_pubs.extend(pubs)
    
    df = pd.DataFrame(all_pubs)
    
    # Convert year to numeric
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    
    return df

print("✓ ScienceDirect loader function defined")

✓ ScienceDirect loader function defined


In [5]:
def load_preprints_data():
    """
    Load preprints CSV data.
    """
    preprints_file = DATA_DIR / 'preprints.csv'
    df = pd.read_csv(preprints_file, encoding='utf-8')
    
    # Standardize column names
    df = df.rename(columns={
        'Title': 'title',
        'Authors': 'authors',
        'Year': 'year',
        'Repository': 'repository',
        'Abstract': 'abstract'
    })
    
    # Add source flags
    df['source'] = 'Preprint'
    df['document_type'] = 'Preprint'
    df['source_file'] = preprints_file.name
    
    # Convert year to numeric
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    
    return df

print("✓ Preprints loader function defined")

✓ Preprints loader function defined


## 3. Load All Data Sources

In [6]:
# Load data from all sources
print("Loading data from all sources...\n")

scopus_df = load_scopus_data()
print(f"✓ Scopus: {len(scopus_df)} records loaded")

sciencedirect_df = load_sciencedirect_data()
print(f"✓ ScienceDirect: {len(sciencedirect_df)} records loaded")

preprints_df = load_preprints_data()
print(f"✓ Preprints: {len(preprints_df)} records loaded")

print(f"\n📊 Total records: {len(scopus_df) + len(sciencedirect_df) + len(preprints_df)}")

Loading data from all sources...

✓ Scopus: 229 records loaded
✓ ScienceDirect: 244 records loaded
✓ Preprints: 27 records loaded

📊 Total records: 500


## 4. Deduplicate Scopus and ScienceDirect (DOI-based Mapping)

Before merging all sources, we identify and remove duplicates between Scopus and ScienceDirect based on DOI matching.

In [7]:
print("="*80)
print("SCOPUS-SCIENCEDIRECT DEDUPLICATION ANALYSIS")
print("="*80)

# Prepare data for comparison
scopus_with_doi = scopus_df[scopus_df['doi'].notna()].copy()
sd_with_doi = sciencedirect_df[sciencedirect_df['doi'].notna()].copy()

print(f"\n📊 Initial Counts:")
print(f"  Scopus total: {len(scopus_df)}")
print(f"  Scopus with DOI: {len(scopus_with_doi)}")
print(f"  ScienceDirect total: {len(sciencedirect_df)}")
print(f"  ScienceDirect with DOI: {len(sd_with_doi)}")

# Find overlapping DOIs
scopus_dois = set(scopus_with_doi['doi'].str.lower().str.strip())
sd_dois = set(sd_with_doi['doi'].str.lower().str.strip())
overlapping_dois = scopus_dois & sd_dois

print(f"\n🔍 Overlap Analysis:")
print(f"  Overlapping DOIs: {len(overlapping_dois)}")
print(f"  Scopus-only DOIs: {len(scopus_dois - sd_dois)}")
print(f"  ScienceDirect-only DOIs: {len(sd_dois - scopus_dois)}")

# Create mapping table
if len(overlapping_dois) > 0:
    print(f"\n📋 Sample of overlapping records:")
    overlap_sample = list(overlapping_dois)[:5]
    for doi in overlap_sample:
        scopus_match = scopus_with_doi[scopus_with_doi['doi'].str.lower().str.strip() == doi]
        sd_match = sd_with_doi[sd_with_doi['doi'].str.lower().str.strip() == doi]
        if len(scopus_match) > 0 and len(sd_match) > 0:
            print(f"\n  DOI: {doi}")
            print(f"    Scopus: {scopus_match.iloc[0]['title'][:60]}...")
            print(f"    ScienceDirect: {sd_match.iloc[0]['title'][:60]}...")
            print(f"    Document Type (SD): {sd_match.iloc[0]['document_type']}")

SCOPUS-SCIENCEDIRECT DEDUPLICATION ANALYSIS

📊 Initial Counts:
  Scopus total: 229
  Scopus with DOI: 216
  ScienceDirect total: 244
  ScienceDirect with DOI: 244

🔍 Overlap Analysis:
  Overlapping DOIs: 0
  Scopus-only DOIs: 214
  ScienceDirect-only DOIs: 244


In [8]:
# Strategy: Keep Scopus records (more complete metadata), remove ScienceDirect duplicates
# But preserve ScienceDirect document type classification

print("\n🔧 Deduplication Strategy:")
print("  - Keep Scopus records (primary source with more metadata)")
print("  - Remove ScienceDirect duplicates")
print("  - Preserve ScienceDirect document type info in Scopus records")

# Normalize DOIs for matching
scopus_df['doi_normalized'] = scopus_df['doi'].str.lower().str.strip()
sciencedirect_df['doi_normalized'] = sciencedirect_df['doi'].str.lower().str.strip()

# Create mapping: DOI -> ScienceDirect document type
sd_doctype_map = sciencedirect_df[sciencedirect_df['doi_normalized'].notna()].set_index('doi_normalized')['document_type'].to_dict()

# Enhance Scopus records with ScienceDirect document type where available
def get_enhanced_doctype(row):
    if pd.notna(row['doi_normalized']) and row['doi_normalized'] in sd_doctype_map:
        return f"{row['document_type']} (SD: {sd_doctype_map[row['doi_normalized']]})"
    return row['document_type']

scopus_df['document_type_enhanced'] = scopus_df.apply(get_enhanced_doctype, axis=1)

# Remove ScienceDirect records that have matching DOIs in Scopus
sd_unique = sciencedirect_df[
    ~sciencedirect_df['doi_normalized'].isin(scopus_df['doi_normalized'].dropna())
].copy()

print(f"\n✅ Deduplication Results:")
print(f"  Scopus records retained: {len(scopus_df)}")
print(f"  ScienceDirect duplicates removed: {len(sciencedirect_df) - len(sd_unique)}")
print(f"  ScienceDirect unique records retained: {len(sd_unique)}")
print(f"  Scopus records enhanced with SD doc type: {scopus_df['document_type_enhanced'].str.contains('SD:', na=False).sum()}")

# Clean up temporary columns
scopus_df = scopus_df.drop('doi_normalized', axis=1)
sd_unique = sd_unique.drop('doi_normalized', axis=1)
sciencedirect_df = sciencedirect_df.drop('doi_normalized', axis=1)


🔧 Deduplication Strategy:
  - Keep Scopus records (primary source with more metadata)
  - Remove ScienceDirect duplicates
  - Preserve ScienceDirect document type info in Scopus records

✅ Deduplication Results:
  Scopus records retained: 229
  ScienceDirect duplicates removed: 0
  ScienceDirect unique records retained: 244
  Scopus records enhanced with SD doc type: 0


## 5. Filter by Year (2010+) and Document Type

Filter all sources to 2010+ (when Decision Intelligence was first developed) and keep only scholarly publications (Article, Review, Conference Paper) for Scopus/ScienceDirect.

In [9]:
print("="*80)
print("FILTERING BY YEAR (2010+) AND DOCUMENT TYPE")
print("="*80)

print(f"\n📊 Before filtering:")
print(f"  Scopus records: {len(scopus_df)}")
print(f"  ScienceDirect unique records: {len(sd_unique)}")
print(f"  Preprints records: {len(preprints_df)}")
print(f"\nYear ranges:")
print(f"  Scopus: {scopus_df['year'].min():.0f} - {scopus_df['year'].max():.0f}")
print(f"  ScienceDirect: {sd_unique['year'].min():.0f} - {sd_unique['year'].max():.0f}")
print(f"  Preprints: {preprints_df['year'].min():.0f} - {preprints_df['year'].max():.0f}")

# Filter by year (2010+) - when Decision Intelligence was first developed
YEAR_THRESHOLD = 2010
print(f"\n🔧 Applying year filter: {YEAR_THRESHOLD}+ (when DI was first developed)")

scopus_df = scopus_df[scopus_df['year'] >= YEAR_THRESHOLD].copy()
sd_unique = sd_unique[sd_unique['year'] >= YEAR_THRESHOLD].copy()
preprints_df = preprints_df[preprints_df['year'] >= YEAR_THRESHOLD].copy()

print(f"\n📊 After year filtering:")
print(f"  Scopus records: {len(scopus_df)}")
print(f"  ScienceDirect records: {len(sd_unique)}")
print(f"  Preprints records: {len(preprints_df)}")

# Define allowed document types for Scopus/ScienceDirect
allowed_types = ['Article', 'Review', 'Conference Paper', 'Conference paper']

print(f"\n🔧 Applying document type filter to Scopus/ScienceDirect")
print(f"  Allowed types: {', '.join(allowed_types)}")

# Filter Scopus and ScienceDirect by document type
scopus_filtered = scopus_df[scopus_df['document_type'].isin(allowed_types)].copy()
sd_filtered = sd_unique[sd_unique['document_type'].isin(allowed_types)].copy()

# Standardize 'Conference paper' to 'Conference Paper'
scopus_filtered['document_type'] = scopus_filtered['document_type'].replace('Conference paper', 'Conference Paper')
sd_filtered['document_type'] = sd_filtered['document_type'].replace('Conference paper', 'Conference Paper')

print(f"\n✅ Final filtering results:")
print(f"  Scopus: {len(scopus_filtered)} records (removed {len(scopus_df) - len(scopus_filtered)} by doc type)")
print(f"  ScienceDirect: {len(sd_filtered)} records (removed {len(sd_unique) - len(sd_filtered)} by doc type)")
print(f"  Preprints: {len(preprints_df)} records (all kept)")

# Update variables for merging
scopus_df = scopus_filtered
sd_unique = sd_filtered

print(f"\n✓ All sources filtered to 2010+")
print(f"✓ Scopus and ScienceDirect filtered to Article, Review, and Conference Paper only")
print(f"✓ All preprints from 2010+ included")

FILTERING BY YEAR (2010+) AND DOCUMENT TYPE

📊 Before filtering:
  Scopus records: 229
  ScienceDirect unique records: 244
  Preprints records: 27

Year ranges:
  Scopus: 1965 - 2026
  ScienceDirect: 1984 - 2026
  Preprints: 2020 - 2026

🔧 Applying year filter: 2010+ (when DI was first developed)

📊 After year filtering:
  Scopus records: 215
  ScienceDirect records: 228
  Preprints records: 27

🔧 Applying document type filter to Scopus/ScienceDirect
  Allowed types: Article, Review, Conference Paper, Conference paper

✅ Final filtering results:
  Scopus: 177 records (removed 38 by doc type)
  ScienceDirect: 228 records (removed 0 by doc type)
  Preprints: 27 records (all kept)

✓ All sources filtered to 2010+
✓ Scopus and ScienceDirect filtered to Article, Review, and Conference Paper only
✓ All preprints from 2010+ included


## 6. Merge All Sources (Filtered Scopus/SD + All Preprints)

In [10]:
# Ensure all dataframes have the same core columns
core_columns = ['title', 'authors', 'year', 'abstract', 'keywords', 'document_type', 'source', 'source_file']

# Add missing columns with None
for df in [scopus_df, sd_unique, preprints_df]:
    for col in core_columns:
        if col not in df.columns:
            df[col] = None

# Add document_type_enhanced to core columns for Scopus
scopus_clean = scopus_df[core_columns + ['doi', 'journal', 'document_type_enhanced']].copy()
sd_clean = sd_unique[core_columns + ['doi', 'journal']].copy()
sd_clean['document_type_enhanced'] = sd_clean['document_type']  # No enhancement for SD-only records

preprints_clean = preprints_df[core_columns + (['repository'] if 'repository' in preprints_df.columns else [])].copy()
preprints_clean['document_type_enhanced'] = preprints_clean['document_type']

# Add repository column to others if not present
if 'repository' not in scopus_clean.columns:
    scopus_clean['repository'] = None
if 'repository' not in sd_clean.columns:
    sd_clean['repository'] = None

# Combine all data (filtered Scopus/SD + all preprints)
dedup_df = pd.concat([scopus_clean, sd_clean, preprints_clean], ignore_index=True)

print(f"✓ Combined dataset: {len(dedup_df)} total records")
print(f"  - Scopus (Article/Review/Conference): {len(scopus_clean)}")
print(f"  - ScienceDirect (Article/Review/Conference): {len(sd_clean)}")
print(f"  - Preprints (all): {len(preprints_clean)}")
print(f"\nYear range: {dedup_df['year'].min():.0f} - {dedup_df['year'].max():.0f}")
print(f"\nData sources:")
print(dedup_df['source'].value_counts())
print(f"\nDocument types:")
print(dedup_df['document_type'].value_counts())

✓ Combined dataset: 432 total records
  - Scopus (Article/Review/Conference): 177
  - ScienceDirect (Article/Review/Conference): 228
  - Preprints (all): 27

Year range: 2010 - 2026

Data sources:
source
ScienceDirect    228
Scopus           177
Preprint          27
Name: count, dtype: int64

Document types:
document_type
Article             273
Conference Paper     80
Review               52
Preprint             27
Name: count, dtype: int64


## 7. Export Integrated Dataset (WITHOUT Abstracts)

**Important**: Abstracts are removed to comply with Git ToU and avoid storing large copyrighted content.

In [12]:
# Create version without abstracts for Git
dedup_df_no_abstract = dedup_df.drop('abstract', axis=1)

# Export the integrated and deduplicated dataset
output_file = OUTPUT_DIR / 'decision_intelligence_integrated.csv'
dedup_df_no_abstract.to_csv(output_file, index=False, encoding='utf-8')

print(f"✓ Integrated dataset exported to: {output_file}")
print(f"  Columns: {', '.join(dedup_df_no_abstract.columns)}")
print(f"  Records: {len(dedup_df_no_abstract)}")
print(f"  Document types: {', '.join(dedup_df_no_abstract['document_type'].unique())}")
print(f"\n⚠️  Note: Abstracts removed for Git compatibility")
print(f"\n✅ Data preparation complete!")
print(f"\n➡️  Next step: Run 02_visualization.ipynb for analysis and visualizations")

✓ Integrated dataset exported to: output/decision_intelligence_integrated.csv
  Columns: title, authors, year, keywords, document_type, source, source_file, doi, journal, document_type_enhanced, repository
  Records: 432
  Document types: Article, Conference Paper, Review, Preprint

⚠️  Note: Abstracts removed for Git compatibility

✅ Data preparation complete!

➡️  Next step: Run 02_visualization.ipynb for analysis and visualizations


## 8. Export Version WITH Abstracts (for Keyword Analysis)

**Note**: This version includes abstracts and is for local keyword analysis only. Do NOT commit to Git.

In [11]:
# Export version WITH abstracts for keyword analysis (local use only)
output_file_with_abstracts = OUTPUT_DIR / 'decision_intelligence_with_abstracts.csv'
dedup_df.to_csv(output_file_with_abstracts, index=False, encoding='utf-8')

print(f"✓ Dataset WITH abstracts exported to: {output_file_with_abstracts}")
print(f"  Records: {len(dedup_df)}")
print(f"  Includes abstracts: {dedup_df['abstract'].notna().sum()} records have abstracts")
print(f"\n⚠️  IMPORTANT: This file is for LOCAL keyword analysis only")
print(f"⚠️  Do NOT commit this file to Git (already in .gitignore)")
print(f"\n➡️  Use this file in 03_keyword_analysis.ipynb for keyword trends and co-occurrence mapping")

✓ Dataset WITH abstracts exported to: output/decision_intelligence_with_abstracts.csv
  Records: 432
  Includes abstracts: 431 records have abstracts

⚠️  IMPORTANT: This file is for LOCAL keyword analysis only
⚠️  Do NOT commit this file to Git (already in .gitignore)

➡️  Use this file in 03_keyword_analysis.ipynb for keyword trends and co-occurrence mapping


## Summary Statistics

In [13]:
print("="*80)
print("DECISION INTELLIGENCE EVOLUTION - DATA PREPARATION SUMMARY")
print("="*80)

print(f"\n📊 DATASET OVERVIEW")
print(f"  Total unique publications: {len(dedup_df)}")
print(f"  Document types: Article, Review, Conference Paper, Preprint")
print(f"  Time span: {dedup_df['year'].min():.0f} - {dedup_df['year'].max():.0f}")

print(f"\n📚 BY SOURCE")
for source, count in dedup_df['source'].value_counts().items():
    pct = count / len(dedup_df) * 100
    print(f"  {source}: {count} ({pct:.1f}%)")

print(f"\n📄 BY DOCUMENT TYPE")
for dtype, count in dedup_df['document_type'].value_counts().items():
    pct = count / len(dedup_df) * 100
    print(f"  {dtype}: {count} ({pct:.1f}%)")

print(f"\n🔍 DATA QUALITY")
print(f"  Records with keywords: {dedup_df['keywords'].notna().sum()} ({dedup_df['keywords'].notna().sum()/len(dedup_df)*100:.1f}%)")
print(f"  Records with DOI: {dedup_df['doi'].notna().sum()} ({dedup_df['doi'].notna().sum()/len(dedup_df)*100:.1f}%)")

print("\n" + "="*80)

DECISION INTELLIGENCE EVOLUTION - DATA PREPARATION SUMMARY

📊 DATASET OVERVIEW
  Total unique publications: 432
  Document types: Article, Review, Conference Paper, Preprint
  Time span: 2010 - 2026

📚 BY SOURCE
  ScienceDirect: 228 (52.8%)
  Scopus: 177 (41.0%)
  Preprint: 27 (6.2%)

📄 BY DOCUMENT TYPE
  Article: 273 (63.2%)
  Conference Paper: 80 (18.5%)
  Review: 52 (12.0%)
  Preprint: 27 (6.2%)

🔍 DATA QUALITY
  Records with keywords: 381 (88.2%)
  Records with DOI: 398 (92.1%)

